In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F, Window

RUN_DAY = F.to_date(F.lit(RUN_DATE))

def prep(table):
    """Read a Bronze table, keep a copy of the original row, then tidy the text."""
    raw = spark.table(f"{BRONZE}.{table}")
    data_cols = [c for c in raw.columns if not c.startswith("_")]
    df = raw.withColumn("_raw", F.to_json(F.struct(*data_cols)))
    for c, t in df.dtypes:
        if t == "string" and c != "_raw":
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))))
    return df

def latest_per_key(df, key):
    """Keep one row per ID: the newest run, then the newest load."""
    w = Window.partitionBy(key).orderBy(F.col("_run_date").desc(), F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def split_good_bad(df, table, key, rules):
    """Run the rules. Return (good rows, quarantined rows with a reason)."""
    checks = [F.when(~F.coalesce(ok, F.lit(False)), F.lit(name)) for name, ok in rules.items()]
    df = df.withColumn("_reject_reason", F.coalesce(*checks))
    good = df.filter("_reject_reason IS NULL").drop("_reject_reason")
    bad = df.filter("_reject_reason IS NOT NULL").select(
        F.lit(table).alias("table_name"),
        F.col(key).alias("business_key"),
        F.col("_reject_reason").alias("reason"),
        F.col("_raw").alias("raw_json"),
        "_run_date",
        F.current_timestamp().alias("_quarantined_at"))
    return good, bad

def write_silver(df, table, key):
    """First run: create the table. Every run after: MERGE (only new or changed rows)."""
    target = f"{SILVER}.{table}"                                            # e.g. workspace.rm_silver.clients
    df = df.drop("_raw")                                                    # Silver doesn't keep the photocopy
    data_cols = [c for c in df.columns if not c.startswith("_")]            # business columns only (not lineage/helpers)
    df = (df.withColumn("_row_hash", F.sha2(F.to_json(F.struct(*data_cols)), 256))  # fingerprint of the business values
            .withColumn("_silver_loaded_at", F.current_timestamp()))        # when this row was written to Silver

    if not spark.catalog.tableExists(target):                               # table not there yet?
        df.write.format("delta").saveAsTable(target)                        #   create it with everything
    else:                                                                   # table already exists?
        df.createOrReplaceTempView("_incoming")                             #   give the new rows a temporary name for SQL
        spark.sql(f"""
            MERGE INTO {target} AS t                                        -- t = what's already in Silver
            USING _incoming AS s                                            -- s = what just arrived
            ON t.{key} = s.{key}                                            -- match rows by their ID
            WHEN MATCHED AND t._row_hash <> s._row_hash THEN UPDATE SET *   -- same ID, different values: update
            WHEN NOT MATCHED THEN INSERT *                                  -- new ID: insert
        """)                                                                # same ID, same fingerprint: do nothing
    return spark.table(target).count()                                      # row count, for the printout

In [0]:
rm = latest_per_key(prep("relationship_managers"), "rm_id")

rm_good, rm_bad = split_good_bad(rm, "relationship_managers", "rm_id", {
    "rm_id missing or malformed": F.col("rm_id").rlike(r"^RM-\d{3}$"),
    "rm_name missing": F.col("rm_name").isNotNull(),
})

In [0]:
cl = (latest_per_key(prep("clients"), "client_id")
      .withColumn("date_of_birth", F.expr("try_cast(date_of_birth AS DATE)"))
      .withColumn("client_since", F.expr("try_cast(client_since AS DATE)"))
      .withColumn("risk_profile_date", F.expr("try_cast(risk_profile_date AS DATE)"))
      .withColumn("monthly_income_zar", F.expr("try_cast(monthly_income_zar AS DECIMAL(18,2))"))
      .withColumn("email", F.lower("email"))
      .join(rm_good.select("rm_id").withColumn("_rm_ok", F.lit(True)), "rm_id", "left"))

cl_good, cl_bad = split_good_bad(cl, "clients", "client_id", {
    "client_id missing or malformed": F.col("client_id").rlike(r"^CL-[0-9a-f]{10}$"),
    "segment not allowed": F.col("segment").isin("Private Banking", "Signature"),
    "risk_profile not allowed": F.col("risk_profile").isin("Conservative", "Moderate", "Aggressive"),
    "income missing or not positive": F.col("monthly_income_zar") > 0,
    "date_of_birth invalid": F.col("date_of_birth") < RUN_DAY,
    "client_since invalid": F.col("client_since") <= RUN_DAY,
    "risk_profile_date invalid": F.col("risk_profile_date") <= RUN_DAY,
    "rm_id not found": F.col("_rm_ok"),
})
cl_good = cl_good.drop("_rm_ok")

In [0]:
print("relationship_managers:", write_silver(rm_good, "relationship_managers", "rm_id"))
print("clients:", write_silver(cl_good, "clients", "client_id"))
rejects = [rm_bad, cl_bad]

In [0]:
valid_clients = spark.table(f"{SILVER}.clients").select("client_id").withColumn("_client_ok", F.lit(True))

ac = (latest_per_key(prep("accounts"), "account_id")
      .withColumn("opened_date", F.expr("try_cast(opened_date AS DATE)"))
      .withColumn("balance", F.expr("try_cast(balance AS DECIMAL(18,2))"))
      .join(valid_clients, "client_id", "left"))

ac_good, ac_bad = split_good_bad(ac, "accounts", "account_id", {
    "account_id missing or malformed": F.col("account_id").rlike(r"^AC-[0-9a-f]{10}$"),
    "product not allowed": F.col("product").isin("Private Banking Account", "Signature Account", "Tax-Free Savings", "Optimum Offshore"),
    "currency not allowed": F.col("currency").isin("ZAR", "USD"),
    "currency does not match product": (F.col("product") == "Optimum Offshore") == (F.col("currency") == "USD"),
    "opened_date invalid": F.col("opened_date") <= RUN_DAY,
    "balance missing or invalid": F.col("balance").isNotNull(),
    "client_id not found": F.col("_client_ok"),
})
print("accounts:", write_silver(ac_good.drop("_client_ok"), "accounts", "account_id"))
rejects.append(ac_bad)

In [0]:
valid_accounts = spark.table(f"{SILVER}.accounts").select("account_id").withColumn("_account_ok", F.lit(True))

cd = (latest_per_key(prep("cards"), "card_id")
      .withColumn("credit_limit_zar", F.expr("try_cast(credit_limit_zar AS DECIMAL(18,2))"))
      .join(valid_accounts, "account_id", "left"))

cd_good, cd_bad = split_good_bad(cd, "cards", "card_id", {
    "card_id missing or malformed": F.col("card_id").rlike(r"^CD-[0-9a-f]{10}$"),
    "card_type not allowed": F.col("card_type").isin("Debit", "Platinum Credit"),
    "last4 not 4 digits": F.col("last4").rlike(r"^\d{4}$"),
    "credit limit does not match card type": F.when(F.col("card_type") == "Debit", F.col("credit_limit_zar").isNull())
                                              .otherwise(F.col("credit_limit_zar") > 0),
    "account_id not found": F.col("_account_ok"),
})
print("cards:", write_silver(cd_good.drop("_account_ok"), "cards", "card_id"))
rejects.append(cd_bad)

In [0]:
ln = (latest_per_key(prep("loans"), "loan_id")
      .withColumn("principal_zar", F.expr("try_cast(principal_zar AS DECIMAL(18,2))"))
      .withColumn("outstanding_zar", F.expr("try_cast(outstanding_zar AS DECIMAL(18,2))"))
      .withColumn("interest_rate_pct", F.expr("try_cast(interest_rate_pct AS DECIMAL(5,2))"))
      .withColumn("start_date", F.expr("try_cast(start_date AS DATE)"))
      .join(valid_clients, "client_id", "left"))

ln_good, ln_bad = split_good_bad(ln, "loans", "loan_id", {
    "loan_id missing or malformed": F.col("loan_id").rlike(r"^LN-[0-9a-f]{10}$"),
    "loan_type not allowed": F.col("loan_type").isin("Home loan", "Vehicle finance"),
    "principal missing or not positive": F.col("principal_zar") > 0,
    "outstanding outside 0 to principal": F.col("outstanding_zar").between(F.lit(0), F.col("principal_zar")),
    "interest rate outside 0 to 30": F.col("interest_rate_pct").between(0, 30),
    "start_date invalid": F.col("start_date") <= RUN_DAY,
    "client_id not found": F.col("_client_ok"),
})
print("loans:", write_silver(ln_good.drop("_client_ok"), "loans", "loan_id"))
rejects.append(ln_bad)

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

valid_cards = spark.table(f"{SILVER}.cards").select("card_id", F.col("account_id").alias("_card_account"))

tx = (latest_per_key(prep("transactions"), "txn_id")
      .withColumn("txn_ts", F.expr("try_cast(txn_ts AS TIMESTAMP)"))
      .withColumn("amount_zar", F.expr("try_cast(amount_zar AS DECIMAL(18,2))"))
      .withColumn("_sast", F.from_utc_timestamp("txn_ts", "Africa/Johannesburg"))
      .withColumn("txn_date_sast", F.to_date("_sast"))
      .withColumn("txn_hour_sast", F.hour("_sast"))
      .drop("_sast")
      .join(valid_accounts, "account_id", "left")
      .join(valid_cards, "card_id", "left"))

card_channels = ["POS", "Online"]
no_card_channels = ["EFT", "Transfer", "Debit order"]

tx_good, tx_bad = split_good_bad(tx, "transactions", "txn_id", {
    "txn_id missing or malformed": F.col("txn_id").rlike(r"^TX-[0-9a-f]{16}$"),
    "txn_ts invalid": F.col("txn_ts").isNotNull(),
    "txn after run date": F.col("txn_date_sast") <= RUN_DAY,
    "amount missing or zero": F.col("amount_zar") != 0,
    "direction not allowed": F.col("direction").isin("debit", "credit"),
    "sign does not match direction": F.when(F.col("direction") == "debit", F.col("amount_zar") < 0)
                                      .otherwise(F.col("amount_zar") > 0),
    "channel not allowed": F.col("channel").isin(card_channels + no_card_channels),
    "card presence does not match channel": F.when(F.col("channel").isin(card_channels), F.col("card_id").isNotNull())
                                             .otherwise(F.col("card_id").isNull()),
    "country_code not 2 letters": F.col("country_code").rlike(r"^[A-Z]{2}$"),
    "merchant_category missing": F.col("merchant_category").isNotNull(),
    "account_id not found": F.col("_account_ok"),
    "card not found or on another account": F.when(F.col("card_id").isNull(), F.lit(True))
                                              .otherwise(F.col("_card_account") == F.col("account_id")),
})
print("transactions:", write_silver(tx_good.drop("_account_ok", "_card_account"), "transactions", "txn_id"))
rejects.append(tx_bad)

In [0]:
from functools import reduce

(reduce(lambda a, b: a.unionByName(b), rejects)
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{SILVER}._quarantine"))

display(spark.table(f"{SILVER}._quarantine").groupBy("table_name", "reason").count())